# Data Structures & Algorithms in Python — Expert Interview Guide

Covers: built-in DS complexity, `heapq`, `bisect`, `deque`, trees, graphs, DP, and classic patterns.

> **Python advantage:** Rich built-in DS with known time complexity. Know WHEN to use each.

## 1. Built-in Complexity — Proven with `timeit`

| Op | list | dict | set |

|---|---|---|---|

| `in` | O(n) | O(1) avg | O(1) avg |

| `append` | O(1) amort | — | — |

| `insert(0,x)` | O(n) | — | — |

In [ ]:
import timeit

setup = 'd = {i: i for i in range(100000)}; lst = list(range(100000))'
dict_t = timeit.timeit('50000 in d', setup=setup, number=100000)
list_t = timeit.timeit('50000 in lst', setup=setup, number=100000)
print(f'dict O(1): {dict_t:.4f}s')
print(f'list O(n): {list_t:.4f}s  ({list_t/dict_t:.0f}x slower)')

# list.append O(1) vs insert(0) O(n)
app_t = timeit.timeit('lst.append(1)', setup='lst=[]', number=100000)
ins_t = timeit.timeit('lst.insert(0,1)', setup='lst=list(range(1000))', number=10000)
print(f'append: {app_t:.4f}s  O(1)')
print(f'insert(0): {ins_t:.4f}s  O(n)')

> **Interview Insight:** `list.pop()` is O(1) from the end but `list.pop(0)` is O(n). Use `collections.deque.popleft()` for O(1) left removal.

## 2. `collections.deque` — O(1) Both Ends

In [ ]:
from collections import deque
import timeit

# O(1) at both ends
dq = deque([1, 2, 3])
dq.appendleft(0); dq.append(4)
print(f'deque: {dq}')
print(f'popleft: {dq.popleft()}')  # O(1)
print(f'pop: {dq.pop()}')          # O(1)

# maxlen circular buffer (sliding window store)
window = deque(maxlen=3)
for x in range(6):
    window.append(x)
    print(f'  add {x}: {list(window)}')

# BFS uses deque as queue
def bfs(graph, start):
    visited, queue, order = {start}, deque([start]), []
    while queue:
        node = queue.popleft()
        order.append(node)
        for nb in graph.get(node, []):
            if nb not in visited:
                visited.add(nb); queue.append(nb)
    return order

g = {'A':['B','C'], 'B':['D'], 'C':['D','E'], 'D':[], 'E':[]}
print(bfs(g, 'A'))

> **Interview Insight:** `deque` is O(n) for random access `dq[i]`. Use it ONLY as a queue/stack. For a sliding window with random access, use a list with index arithmetic.

## 3. `heapq` — Min-Heap / Priority Queue

In [ ]:
import heapq

nums = [3, 1, 4, 1, 5, 9, 2, 6]
heapq.heapify(nums)  # in-place O(n)
print(f'heap: {nums}')
print(f'smallest: {heapq.heappop(nums)}')

print(f'3 largest: {heapq.nlargest(3, [3,1,4,1,5,9,2,6])}')
print(f'3 smallest: {heapq.nsmallest(3, [3,1,4,1,5,9,2,6])}')

# Max-heap: negate values
max_heap = []
for n in [3, 1, 4, 1, 5, 9]:
    heapq.heappush(max_heap, -n)
print('Max order:', end=' ')
while max_heap: print(-heapq.heappop(max_heap), end=' ')
print()

# Priority queue with (priority, item) tuples
tasks = []
heapq.heappush(tasks, (3, 'low'))
heapq.heappush(tasks, (1, 'urgent'))
heapq.heappush(tasks, (2, 'normal'))
while tasks:
    p, task = heapq.heappop(tasks)
    print(f'  [{p}] {task}')

# K-th largest
def kth_largest(nums, k):
    h = []
    for n in nums:
        heapq.heappush(h, n)
        if len(h) > k: heapq.heappop(h)
    return h[0]

print(f'2nd largest in [3,2,1,5,6,4]: {kth_largest([3,2,1,5,6,4], 2)}')

> **Interview Insight:** `heapq.nlargest(k, data)` is O(n log k) — faster than `sorted()` O(n log n) when k << n. For k close to n, prefer `sorted()[-k:]`.

## 4. `bisect` — Binary Search on Sorted Lists

In [ ]:
import bisect

sl = [1, 3, 5, 7, 9, 11]
print(f'bisect_left(5):  {bisect.bisect_left(sl, 5)}')
print(f'bisect_right(5): {bisect.bisect_right(sl, 5)}')

bisect.insort(sl, 4)  # inserts and keeps sorted
print(f'after insort(4): {sl}')

# Contains check O(log n)
def contains(lst, val):
    i = bisect.bisect_left(lst, val)
    return i < len(lst) and lst[i] == val

print(f'contains 5: {contains(sl, 5)}')
print(f'contains 6: {contains(sl, 6)}')

# Grade lookup via bisect
def grade(score):
    return 'FDCBA'[bisect.bisect([60,70,80,90], score)]

for s in [45, 65, 75, 88, 95]:
    print(f'  {s}: {grade(s)}')

> **Interview Insight:** `bisect.insort` is O(log n) to find position but O(n) to insert (list shifting). For frequent insertions into sorted order, use `sortedcontainers.SortedList` which is O(log n) overall.

## 5. Linked List — Reverse & Cycle Detection (Floyd's)

In [ ]:
class ListNode:
    def __init__(self, val=0, nxt=None): self.val=val; self.next=nxt

def make_list(vals):
    head = None
    for v in reversed(vals): head = ListNode(v, head)
    return head

def to_list(head, limit=20):
    res, seen = [], set()
    while head and id(head) not in seen and len(res) < limit:
        seen.add(id(head)); res.append(head.val); head = head.next
    return res

def reverse_list(head):
    prev, curr = None, head
    while curr:
        nxt = curr.next; curr.next = prev
        prev = curr; curr = nxt
    return prev

lst = make_list([1,2,3,4,5])
print(f'Original: {to_list(lst)}')
print(f'Reversed: {to_list(reverse_list(lst))}')

# Floyd's cycle detection
def has_cycle(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next; fast = fast.next.next
        if slow is fast: return True
    return False

# Build cycle: 1->2->3->4->2 (cycle at 2)
n1,n2,n3,n4 = ListNode(1),ListNode(2),ListNode(3),ListNode(4)
n1.next=n2; n2.next=n3; n3.next=n4; n4.next=n2
print(f'has_cycle: {has_cycle(n1)}')
print(f'no_cycle:  {has_cycle(make_list([1,2,3]))}')

> **Interview Insight:** Floyd's algorithm: slow moves 1 step, fast moves 2 steps. They meet inside the cycle. Then move one pointer to head and advance both at speed 1 — they meet at the **cycle entry point**.

## 6. Binary Tree — Traversal & BFS

In [ ]:
from collections import deque

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val=val; self.left=left; self.right=right

def build(vals):
    if not vals: return None
    root = TreeNode(vals[0])
    q = deque([root]); i = 1
    while q and i < len(vals):
        node = q.popleft()
        if i < len(vals) and vals[i] is not None:
            node.left = TreeNode(vals[i]); q.append(node.left)
        i += 1
        if i < len(vals) and vals[i] is not None:
            node.right = TreeNode(vals[i]); q.append(node.right)
        i += 1
    return root

def inorder(n):   return inorder(n.left)+[n.val]+inorder(n.right) if n else []
def preorder(n):  return [n.val]+preorder(n.left)+preorder(n.right) if n else []
def postorder(n): return postorder(n.left)+postorder(n.right)+[n.val] if n else []

def level_order(root):
    if not root: return []
    res, q = [], deque([root])
    while q:
        level = [q.popleft() for _ in range(len(q)) if True]
        # fix: proper level order
        pass
    return res

def bfs_levels(root):
    if not root: return []
    result, q = [], deque([root])
    while q:
        lvl = []
        for _ in range(len(q)):
            n = q.popleft(); lvl.append(n.val)
            if n.left: q.append(n.left)
            if n.right: q.append(n.right)
        result.append(lvl)
    return result

def height(n): return 0 if not n else 1+max(height(n.left), height(n.right))

root = build([1,2,3,4,5,6,7])
print(f'Inorder:     {inorder(root)}')
print(f'Preorder:    {preorder(root)}')
print(f'Postorder:   {postorder(root)}')
print(f'BFS levels:  {bfs_levels(root)}')
print(f'Height:      {height(root)}')

> **Interview Insight:** Inorder traversal of a **BST** yields sorted order. Preorder is useful for serializing a tree. Postorder for deletion (delete children before parent).

## 7. Dynamic Programming — Memoization, Tabulation, Patterns

In [ ]:
import functools

# Fibonacci: naive O(2^n) -> memoized O(n)
@functools.lru_cache(maxsize=None)
def fib_memo(n): return n if n < 2 else fib_memo(n-1)+fib_memo(n-2)
print(f'fib(50) = {fib_memo(50)}')
print(f'cache: {fib_memo.cache_info()}')

# Bottom-up tabulation: O(n) time, O(1) space
def fib_tab(n):
    if n < 2: return n
    a, b = 0, 1
    for _ in range(2, n+1): a, b = b, a+b
    return b
print(f'fib_tab(50) = {fib_tab(50)}')

# Sliding window — max sum subarray of size k
def max_window(arr, k):
    s = sum(arr[:k]); best = s
    for i in range(k, len(arr)):
        s += arr[i] - arr[i-k]; best = max(best, s)
    return best
print(f'max_window([2,1,5,1,3,2], k=3) = {max_window([2,1,5,1,3,2], 3)}')

# Two pointers — pair sum in sorted array
def two_sum(arr, target):
    l, r = 0, len(arr)-1
    while l < r:
        s = arr[l]+arr[r]
        if s == target: return (arr[l], arr[r])
        elif s < target: l += 1
        else: r -= 1
    return None
print(f'two_sum([1,2,3,4,6], 6) = {two_sum([1,2,3,4,6], 6)}')

# 0/1 Knapsack
def knapsack(weights, values, cap):
    n = len(weights)
    dp = [[0]*(cap+1) for _ in range(n+1)]
    for i in range(1, n+1):
        for w in range(cap+1):
            dp[i][w] = dp[i-1][w]
            if weights[i-1] <= w:
                dp[i][w] = max(dp[i][w], dp[i-1][w-weights[i-1]]+values[i-1])
    return dp[n][cap]
print(f'knapsack(cap=8): {knapsack([2,3,4,5],[3,4,5,6],8)}')

> **Interview Insight:** Always explain the DP recurrence before coding. Start recursive with `lru_cache` (easiest to reason about), then optimize to tabulation if needed. Time: O(n*cap), Space: O(cap) with rolling array.